In [ ]:
# Cleaned & anonymized script — replace /path/to/your/CausalICM/ with your local folder
import sys
# <- Update this to point to your local CausalICM repository root (not a __pycache__ folder)
sys.path.append("/path/to/your/CausalICM/")
from CausalICM.functions.data_simulation_multivariate import data_simulation1

# reproducibility
seed_value = 1234
import os
os.environ['PYTHONHASHSEED'] = str(seed_value)

import random
random.seed(seed_value)

import numpy as np
np.random.seed(seed_value)

# simulate multivariate data
data_exp, data_obs = data_simulation1(1000, beta0=-10.0, beta1=-8.0, beta2=-8.0)
data_full = np.r_[data_exp, data_obs]

# model for probability of participation
from sklearn import linear_model
logr = linear_model.LogisticRegression()
logr.fit(data_full[:, 1:4], data_full[:, 0])
prob = logr.predict_proba(data_full[:, 1:4])
prob_part = prob[:, 1]
prob_nonpart = prob[:, 0]

# add inverse-probability weight column (1 / Pr[S=1])
data_full = np.c_[data_full, 1 / prob_part]


# cross-validation to tune rho
from sklearn.model_selection import KFold
import numpy.random as nprand
from CausalICM.functions.model_training import ICM
from CausalICM.functions.weightedMSE import weighted_mean_squared_error

seed_value = 1234

rho_values = np.round(np.arange(0, 1.05, 0.1), 1)
num_folds = 5

min_avg_wmse = float('inf')
best_rho = None
wmse_per_rho = {}

kf = KFold(n_splits=num_folds, shuffle=True, random_state=seed_value)

for rho in rho_values:
    print(f"Rho: {rho}")
    avg_wmse = 0
    fold_num = 0
    wmse_per_fold = []
    
    for train_index, val_index in kf.split(data_full):
        train_data, val_data = data_full[train_index], data_full[val_index]
        # validate only on experimental units
        val_data = val_data[val_data[:, 0] == 1, :]
        if val_data.shape[0] == 0:
            # skip fold if no experimental units in validation (rare but safe)
            continue
        
        X_val = val_data[:, 1:6]
        T_val = val_data[:, 6]
        Y_val = val_data[:, 7]
        w = val_data[:, 8]
        
        m0, m1 = ICM(
            X_E = train_data[train_data[:, 0] == 1, 1:6],
            X_O = train_data[train_data[:, 0] == 0, 1:6],
            T_E = train_data[train_data[:, 0] == 1, 6],
            T_O = train_data[train_data[:, 0] == 0, 6],
            Y_E = train_data[train_data[:, 0] == 1, 7],
            Y_O = train_data[train_data[:, 0] == 0, 7],
            r = 2, ID = 5, AD = [0,1,2,3,4], rho = rho
        )

        # prepare inputs for predict_noiseless (T=0 for control, T=1 for treated)
        X_input = val_data[:, 1:6]
        predY0_exp = m0.predict_noiseless(np.c_[X_input, np.zeros(X_input.shape[0])])
        predY1_exp = m1.predict_noiseless(np.c_[X_input, np.ones(X_input.shape[0])])
        
        predCATE_exp = predY1_exp[0] - predY0_exp[0]
        varCATE_exp = predY1_exp[1] + predY0_exp[1]

        wmse = weighted_mean_squared_error(weight = w, t = T_val, y_true = Y_val,
                                           y_pred0 = predY0_exp[0], y_pred1 = predY1_exp[0])
        avg_wmse += wmse
        wmse_per_fold.append(wmse)

        print(f" Fold {fold_num + 1} WMSE: {wmse}")
        fold_num += 1
    
    # normalize by number of folds actually used
    if fold_num > 0:
        avg_wmse /= fold_num
    else:
        avg_wmse = float('inf')

    print(f" Average WMSE for rho {rho}: {avg_wmse}")
    wmse_per_rho[rho] = wmse_per_fold

    if avg_wmse < min_avg_wmse:
        min_avg_wmse = avg_wmse
        best_rho = rho

print(f"Best rho: {best_rho}, Minimum Avg WMSE: {min_avg_wmse}")

# save chosen rho
import pandas as pd
sim1_best_rho_df = pd.DataFrame([best_rho], columns=['best_rho'])
sim1_best_rho_df.to_csv('sim1_best_rho.csv', index=False)